# COMMUN

### Imports et configuration

In [1]:
import os
import yaml
import pandas as pd
import numpy as np
import requests
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from pathlib import Path
from sqlalchemy.exc import SQLAlchemyError

### Charger variables d'environnement depuis .env

In [2]:
load_dotenv()

True

In [3]:
try:
    ROOT_DIR = Path(__file__).resolve().parents[1]
except NameError:
    ROOT_DIR = Path.cwd().parent

CONFIG_PATH = ROOT_DIR / "config.yml"
SQL_FILES_PATH = ROOT_DIR / "etl"
print(CONFIG_PATH)
print(SQL_FILES_PATH)

/home/samchaka/simplon/Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes/config.yml
/home/samchaka/simplon/Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes/etl


### Définir le chemin racine du projet (quel que soit le dossier courant)

In [4]:
def load_config(path):
    with open(path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
        
    for section, values in config.items():
        for key, val in values.items():
            if isinstance(val, str) and val.startswith("${"):
                env_var = val.strip("${}")
                config[section][key] = os.getenv(env_var)    
    return config

### Lecture du fichier config.yml

In [5]:
conf = load_config(CONFIG_PATH)
db_conf = conf['database']
sql_file_conf = conf['sqlfile']

NameError: name 'load_config' is not defined

# AHMED

In [9]:
def test_postgres_connection(db_config: dict, db_name: str | None = None) -> str:
    """
    Teste la connexion à une base PostgreSQL et retourne un message lisible.

    Args:
        db_config (dict): Dictionnaire contenant les informations de connexion :
            - user : nom d'utilisateur PostgreSQL
            - password : mot de passe
            - host : adresse du serveur
            - port : port PostgreSQL
            - db_default : base par défaut (souvent 'postgres')
        db_name (str | None): Nom de la base à tester. Si None, utilise db_config['db_default'].

    Returns:
        str: Message de succès avec la version PostgreSQL ou message d'erreur.
    """

    # Choix de la base : soit celle fournie, soit la base par défaut
    db_to_use = db_name or db_config["db_default"]

    # Création de l'URL de connexion PostgreSQL compatible SQLAlchemy
    db_url = (
        f"postgresql+psycopg2://{db_config['user']}:{db_config['password']}"
        f"@{db_config['host']}:{db_config['port']}/{db_to_use}"
    )

    try:
        # Création de l'engine SQLAlchemy
        engine = create_engine(db_url)

        # Ouverture de la connexion
        with engine.connect() as conn:
            # Exécution d'une requête pour récupérer la version PostgreSQL
            version = conn.execute(text("SELECT version();")).scalar()

            # Message de succès
            message = (
                f"Connexion réussie à la base '{db_to_use}'."
                f"Version PostgreSQL : {version}"
            )
            return message

    except SQLAlchemyError as e:
        # Gestion des erreurs de connexion
        message = (
            f"Erreur de connexion à la base '{db_to_use}'."
            f"{e}"
        )
        return message

    finally:
        # Fermeture propre de l'engine
        if 'engine' in locals():
            engine.dispose()

# execution
test_postgres_connection(db_conf)

In [ ]:
def create_database(db_config: dict, db_name: str):
    """
    Vérifie si une base PostgreSQL existe et la crée si nécessaire.

    Args:
        db_config (dict): Dictionnaire contenant les informations de connexion :
            - user : nom d'utilisateur PostgreSQL
            - password : mot de passe
            - host : adresse du serveur
            - port : port PostgreSQL
            - db_default : base par défaut utilisée pour se connecter initialement
        db_name (str): Nom de la base à créer ou vérifier.

    Returns:
        None
    """

    # Construction de l'URL de connexion sur la base par défaut (souvent 'postgres')
    db_url = (
        f"postgresql+psycopg2://{db_config['user']}:{db_config['password']}"
        f"@{db_config['host']}:{db_config['port']}/{db_config['db_default']}"
    )
    try:
        # Création de l'engine avec autocommit pour exécuter CREATE DATABASE
        engine = create_engine(db_url, isolation_level="AUTOCOMMIT")

        with engine.connect() as conn:
            # Vérifier si la base existe déjà
            result = conn.execute(
                text("SELECT 1 FROM pg_database WHERE datname = :dbname"),
                {"dbname": db_name}
            )
            exists = result.scalar()  # Récupère le premier résultat (1 si la base existe)

            if not exists:
                # Crée la base si elle n'existe pas
                conn.execute(text(f'CREATE DATABASE "{db_name}"'))
                print(f"Base '{db_name}' créée.")
            else:
                print(f"Base '{db_name}' existe déjà.")
    except SQLAlchemyError as e:
        # Gestion des erreurs SQLAlchemy
        print(f"Erreur lors de la vérification ou création de la base : {e}")

    finally:
        # Fermeture propre de l'engine
        if 'engine' in locals():
            engine.dispose()


# execution
create_database(db_conf, db_conf["db_accm"])

In [ ]:
def execute_sql_file(db_conf: dict, db_name: str, sql_file_path: str) -> str:

    db_to_use = db_name
    sql_file = Path(sql_file_path)
    if not sql_file.is_file():
        return f"Fichier SQL introuvable : {sql_file_path}"
    db_url = (
        f"postgresql+psycopg2://{db_conf['user']}:{db_conf['password']}"
        f"@{db_conf['host']}:{db_conf['port']}/{db_to_use}"
    )
    try:
        engine = create_engine(db_url, isolation_level="AUTOCOMMIT")

        # Lire le contenu du fichier SQL
        sql_commands = sql_file.read_text(encoding="utf-8")

        with engine.connect() as conn:
            conn.execute(text(sql_commands))
        return f"Fichier SQL '{sql_file_path}' exécuté avec succès sur la base '{db_to_use}'."
    except SQLAlchemyError as e:
        return f"Erreur lors de l'exécution du fichier SQL : {e}"
    finally:
        if 'engine' in locals():
            engine.dispose()
execute_sql_file(db_conf, db_conf["db_accm"], SQL_FILES_PATH/sql_file_conf["file_1"])

In [ ]:
def execute_sql_to_df(db_conf: dict, db_name: str, sql_file_path: str) -> pd.DataFrame:
    """
    Exécute une requête SQL depuis un fichier sur une base PostgreSQL et retourne le résultat sous forme de DataFrame.

    Args:
        db_conf (dict): Dictionnaire contenant les informations de connexion :
            - user : nom d'utilisateur PostgreSQL
            - password : mot de passe
            - host : adresse du serveur
            - port : port PostgreSQL
            - db_default : base par défaut (optionnelle)
        db_name (str): Nom de la base de données sur laquelle exécuter la requête.
        sql_file_path (str): Chemin vers le fichier SQL contenant la ou les requêtes.

    Returns:
        pd.DataFrame: DataFrame contenant le résultat de la requête si des colonnes existent,
                      sinon un DataFrame avec un message d'information.
    """
    # Convertit le chemin du fichier SQL en objet Path pour faciliter les manipulations
    sql_file = Path(sql_file_path)

    # Vérifie que le fichier SQL existe, sinon retourne un DataFrame avec message d'erreur
    if not sql_file.is_file():
        print(f"Fichier SQL introuvable : {sql_file_path}")
        return pd.DataFrame({"info_message": [f"Fichier SQL introuvable : {sql_file_path}"]})

    # Construction de l'URL de connexion PostgreSQL compatible SQLAlchemy
    connection_url = (
        f"postgresql+psycopg2://{db_conf['user']}:{db_conf['password']}"
        f"@{db_conf['host']}:{db_conf['port']}/{db_name}"
    )
    try:
        # Création de l'objet engine SQLAlchemy avec autocommit pour exécuter DDL si nécessaire
        engine = create_engine(connection_url, isolation_level="AUTOCOMMIT")

        # Lecture du contenu du fichier SQL
        sql_text = sql_file.read_text(encoding="utf-8")

        # Ouverture d'une connexion à la base de données
        with engine.connect() as conn:
            # Exécution de la requête SQL
            result = conn.execute(text(sql_text))
            
            # Si la requête renvoie des lignes (ex: SELECT), créer un DataFrame
            if result.returns_rows:
                df = pd.DataFrame(result.fetchall(), columns=result.keys())
            else:
                # Si la requête ne renvoie rien (ex: CREATE TABLE), renvoyer un message
                df = pd.DataFrame({"info_message": ["Requête exécutée avec succès, pas de résultat à afficher."]})
            
            return df

    except SQLAlchemyError as e:
        # Capture des erreurs SQLAlchemy et retour d'un DataFrame contenant l'erreur
        return pd.DataFrame({"info_message": [f"Erreur lors de l'exécution : {e}"]})

    finally:
        # Libération des ressources de l'engine pour fermer proprement la connexion
        if 'engine' in locals():
            engine.dispose()


## Exemple

df = execute_sql_to_df(db_conf, db_conf["db_accm"], SQL_FILES_PATH/sql_file_conf["file_1"])
df.head()

# ROMAIN

In [ ]:
API_CONFIG = {
    'base_url' : 'https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/accidents-corporels-de-la-circulation-millesime/records',
    'limit_per_request' : 100,
    'max_records' : 1000,
    'timeout' : 30
}

print(f"API: {API_CONFIG['base_url'][:70]}...")

In [ ]:
def extraire_accidents_api(max_records=None):
    """
    Fonction pour extraire les accidents depuis l'API
    
    Paramètres:
        max_records (int): Le nombre maximum d'accidents à extraire. Si None, tous les accidents seront extraits.
    
    Return:
        list: Une liste de dictionnaires représentant les accidents extraîts.
    """

print("=" * 80)
print("EXTRACTION DES DONNÉES")
print("=" * 80)

all_records = []
offset = 0
limit = API_CONFIG['limit_per_request']

try:
    #première requête pour connaitre le total
    print("Récupération du nombre total...")
    response = requests.get(
        API_CONFIG['base_url'],
        params={'limit': 1},
        timeout=API_CONFIG['timeout']
    )
    response.raise_for_status()
    data = response.json()
    total_count = data.get('total_count', 0)
    
    print(f"Total disponible: {total_count:,} enregistrements")
    print(data)       
except requests.RequestException as e:
    print(f"\n✗ Erreur lors de l'extraction: {e}")
    raise

In [ ]:
"""Téléchargement minimaliste du dataset accidents corporels depuis OpenDataSoft.

Version simplifiée sans retry, sans barre de progression, sans validation.
Télécharge le CSV par chunks et le sauvegarde dans data/accidents_corporels_millesime.csv

Usage:
    python scripts/sauvegarde_csv_api_v1.py

Source:
    https://public.opendatasoft.com - Dataset accidents corporels de la circulation
"""

import sys
from dataclasses import dataclass
from pathlib import Path
import requests



@dataclass(frozen=True)
class Config:
    """Configuration du téléchargement."""
    
    base_url: str = "https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets"
    dataset_id: str = "accidents-corporels-de-la-circulation-millesime"
    output_dir: str = "data"
    output_file: str = "accidents_corporels_millesime.csv"
    delimiter: str = ","
    chunk_size: int = 65536 #on lit 64KB par 64KB pour ne pas que Lounes voit la RAM de son pc bruler
    timeout: int = 30


def build_url(config: Config) -> str:
    """Construit l'URL de téléchargement."""
    return f"{config.base_url}/{config.dataset_id}/exports/csv?delimiter={config.delimiter}"


def resolve_path(config: Config) -> Path:
    """Détermine le chemin de sortie."""
    #script_dir = ROOT_DIR
    return ROOT_DIR / config.output_dir / config.output_file


def download_csv(url: str, destination: Path, config: Config) -> None:
    """Télécharge le CSV par chunks."""
    print(f"Téléchargement depuis OpenDataSoft...")

    response = requests.get(url, stream=True, timeout=config.timeout)
    response.raise_for_status()

    destination.parent.mkdir(parents=True, exist_ok=True)

    with response, open(destination, "wb") as handle:
        for chunk in response.iter_content(chunk_size=config.chunk_size):
            if chunk:
                handle.write(chunk)

    print(f"Fichier sauvegardé: {destination}")


def main() -> None:
    """Point d'entrée principal."""
    config = Config()
    url = build_url(config)
    path = resolve_path(config)

    download_csv(url, path, config)

    print("Téléchargement terminé")


if __name__ == "__main__":
    main()


In [11]:
# Import du script avec chemin absolu
import sys

# Chemin absolu vers le dossier scripts
scripts_dir = ROOT_DIR / 'etl'

# Ajouter au path Python
if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)

# Import
from sauvegarde_csv_api import download_accidents

# Téléchargement
download_accidents(
    output=ROOT_DIR / 'data/accidents_corporels_millesime.csv',
    delimiter=',',
    show_progress=True,
    retries=3,
    #limit = 1000,
    #where ="an=2017 AND dep='60'"
)

print(f"✅ Téléchargement réussi !")
from sauvegarde_csv_api import collect_csv_stats
from sauvegarde_csv_api import print_summary

csv_path = ROOT_DIR / "data/accidents_corporels_millesime.csv"
stats = collect_csv_stats(csv_path, delimiter=",")

print(f"✅ Fichier prêt : {stats.record_count:,} accidents")
print_summary(stats)


🚗 TÉLÉCHARGEMENT DATASET ACCIDENTS CORPORELS
📍 Source: OpenDataSoft
📅 Période: 2012-2019
📄 Fichier: /home/samchaka/simplon/Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes/data/accidents_corporels_millesime.csv
🔤 Séparateur: ','

🌐 Connexion à OpenDataSoft...
📥 Téléchargement (taille inconnue)


Téléchargement: 346MB [00:02, 137MB/s]  


✅ Fichier sauvegardé: /home/samchaka/simplon/Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes/data/accidents_corporels_millesime.csv
⏱️  Temps de téléchargement: 2.8 secondes
✅ Téléchargement réussi !
✅ Fichier prêt : 475,911 accidents

📊 RÉSUMÉ DU TÉLÉCHARGEMENT
✅ Fichier valide
   Taille: 346.13 MB (362,941,194 bytes)
   Lignes: 475,912 (incluant l'en-tête)
   Records: 475,911 accidents
   Colonnes: 69

📋 Premières colonnes:
   1. num_acc
   2. datetime
   3. nom_com
   4. an
   5. mois
   6. jour
   7. hrmn
   8. lum
   9. agg
   10. int
   ... et 59 autres colonnes



# LOUNES

In [ ]:
# Importation des données source dans un DataFrame pandas

df_source = pd.read_csv(ROOT_DIR / 'data/accidents_corporels_millesime.csv', delimiter=',', low_memory=False)
df_source.head()

In [ ]:
# TEST / EXPLORATION
# Exploration de la donnée source csv

df_source.info()

In [ ]:
"""
Préparation des données : nettoyage, transformation

Objectif : diviser le csv en 5 tables distinctes (accident, vehicule, usager, lieux, date)
Chaque table sera dans une dataframe pandas distincte

1ere étape : premier petit nettoyage et uniformisation des données
2eme étape : séparation en 5 dataframes
3ème étape : explosion des colonnes multi-valeurs pour les dataframes vehicule et usager
4eme étape : nettoyage spécifique à chaque dataframe
5ème étape : mapping des valeurs catégorielles pour chaque dataframe
6ème étape : tests de validation des données
7ème étape : export des dataframes nettoyés et validés dans postgres
"""
# FONCTION 1 (première transfo sur les noms de colonnes et datetime + création des 5 df source)
# Premier nettoyage et uniformisation des données
df_source.columns = df_source.columns.str.lower().str.strip()
df_source["datetime"] = pd.to_datetime(df_source["datetime"], errors="coerce")
df_source.info()

In [ ]:
# FONCTION 1 - Séparation en 5 dataframes
# 1. Accidents
df_accidents = df_source[[
    'num_acc', 
    'lum', 
    'agg',
    'int',
    'atm',
    'adr',
    'col',
    'circ',
    'plan',
    'prof',
    'surf',
    'infra',
    'situ',
    'year_georef'
]].copy()

# EXPLORATION
# df_accidents.head()
df_accidents.info()

In [ ]:
# EXPLORATION
# Tests de validation dataframe df_accidents

# Vérification des doublons sur 'num_acc
duplicates_accidents = df_accidents.duplicated(subset=['num_acc']).sum()
print(f"Doublons dans df_accidents sur 'num_acc': {duplicates_accidents} \n")

# Vérification des valeurs uniques sur les catégories
for col in ['lum', 'agg', 'int', 'atm', 'col', 'circ', 'plan', 'prof', 'surf', 'infra', 'situ']:
    unique_values = df_accidents[col].unique()
    print(f"\n Valeurs uniques dans '{col}': {unique_values}")

# Compter le nombre de 'nan' et '-1' dans chaque colonne catégorielle
for col in ['lum', 'agg', 'int', 'atm', 'col', 'circ', 'plan', 'prof', 'surf', 'infra', 'situ']:
    nan_count = df_accidents[col].isna().sum()
    neg_one_count = (df_accidents[col] == '-1').sum()
    print(f"\n Colonne '{col}': NaN = {nan_count}, -1 = {neg_one_count}")


In [ ]:
# FONCTION 2 - TRANSFORMATION DF ACCIDENTS
# Transformation des valeurs '-1' en NaN pour les colonnes catégorielles
for col in ['lum', 'agg', 'int', 'atm', 'col', 'circ', 'plan', 'prof', 'surf', 'infra', 'situ']:
    df_accidents[col] = df_accidents[col].replace('-1', np.nan)

In [ ]:
# FONCTION 2 - TRANSFORMATION DF ACCIDENTS
# Nettoyer "infra" et "situ" car ont des numéros écris en type object et des catégories mélangées. Transformer les numéros en NaN
df_accidents['infra'] = df_accidents['infra'].apply(lambda x: np.nan if str(x).isdigit() else x)
df_accidents['situ'] = df_accidents['situ'].apply(lambda x: np.nan if str(x).isdigit() else x)

In [ ]:
# La dataframe df_accidents est prête pour export. Mapping catégories à réaliser.



In [ ]:
# FONCTION 1 - Séparation en 5 dataframes
# 2. Lieux
df_lieux = df_source[[
    'com_code',
    'com_name',
    'dep_code',
    'dep_name',
    'reg_code', 
    'reg_name', 
    'epci_code', 
    'epci_name',
    'lat', 
    'long', 
    'catr', 
    'v1', 
    'voie', 
    'v2', 
    'nbv', 
    'vosp', 
    'pr', 
    'pr1', 
    'lartpc', 
    'larrout', 
    'num_acc'
]].copy()

# EXPLORATION
# df_lieux.head()
df_lieux.info()

In [ ]:
# EXPLORATION
# Vérification des doublons sur 'num_acc
duplicates_lieux = df_lieux.duplicated(subset=['num_acc']).sum()
print(f"Doublons dans df_lieux sur 'num_acc': {duplicates_lieux} \n")

# Vérification des catégories
# df_lieux['catr'].value_counts(dropna=False)

# Notes nettoyage : 
# drop 'voie', drop 'V1', drop 'V2', drop 'pr', drop 'pr1', drop 'lartpc', drop 'larrout, drop 'epci_code', drop 'epci_name', drop 'lat', drop 'long
# numéros à mettre en NaN dans 'catr'
# nbv > 6 ou nbv = -1 ou nbv = 0 --> NaN (explication : deux fois 3 voies sur autoroute est le max selon nous soit 6 voies)
# vosp = -1 --> NaN

# FONCTION 3 - TRANSFORMATION DF LIEUX
# Drop des colonnes inutiles
df_lieux = df_lieux.drop(columns=['voie', 'v1', 'v2', 'pr', 'pr1', 'lartpc', 'larrout', 'epci_code', 'epci_name', 'lat', 'long'])


In [ ]:
# FONCTION 3 - TRANSFORMATION DF LIEUX
# Nettoyer "catr"
df_lieux['catr'] = df_lieux['catr'].apply(lambda x: 'autre' if str(x).isdigit() else x)

df_lieux['catr'].value_counts(dropna=False)

In [ ]:
# FONCTION 3 - TRANSFORMATION DF LIEUX
# Nettoyer 'nbv'
# nbv > 6 ou nbv = -1 ou nbv = 0 --> NaN (explication : deux fois 3 voies sur autoroute est le max selon nous soit 6 voies). values are float.
df_lieux['nbv'] = df_lieux['nbv'].apply(lambda x: np.nan if (x > 6.0 or x == 0.0) or x == -1.0 else x)

# transformer le dtype en int
df_lieux['nbv'] = df_lieux['nbv'].astype('Int64')

In [ ]:
# FONCTION 3 - TRANSFORMATION DF LIEUX
# Nettoyer 'vosp'
# vosp = -1 --> NaN
df_lieux['vosp'] = df_lieux['vosp'].apply(lambda x: np.nan if x == '-1' else x)
df_lieux['vosp'].value_counts(dropna=False)

In [ ]:
# EXPLORATION
# Last check df_lieux
df_lieux.info()

# df_lieux est prêt pour export dans postgres

In [ ]:
# FONCTION 1 - Séparation en 5 dataframes

# 3. Date_accident
df_date_accident = df_source[[
    'datetime', 
    'an', 
    'mois', 
    'jour', 
    'hrmn', 
    'num_acc'
]].copy()

# EXPLORATION
# df_date_accident.head()
df_date_accident.info()

In [ ]:
# EXPLORATION
# Vérification des colonnes 'an', 'mois', 'jour' pour éviter des aberrations
df_date_accident[['an', 'mois', 'jour']].describe()

# FONCTION 4 - TRANSFORMATION DF DATE_ACCIDENT
# Vérification et transformation du dtype de la colonne 'hrmn' (exemple : dtype object "16:45") en datetime object "HH:MM"
df_date_accident['hrmn'] = pd.to_datetime(df_date_accident['hrmn'], format='%H:%M', errors='coerce').dt.time

In [ ]:
# EXPLORATION
# Last check for df_date_accident
df_date_accident.info()

# df_date_accident est prêt pour export dans postgres

In [ ]:
# FONCTION 1 - Séparation en 5 dataframes
 
# 4. Vehicules
df_vehicules = df_source[[
    'num_veh', 
    'catv', 
    'choc', 
    'senc', 
    'obs', 
    'obsm', 
    'occutc', 
    'manv', 
    'num_acc'
]].copy()

# EXPLORATION
df_vehicules.info()

In [ ]:
# EXPLORATION
df_vehicules.head(20)

In [ ]:
# EXPLORATION
# Transformer les colonnes qui contiennent plusieurs valeurs séparées par des virgules en listes
veh_cols = ['num_veh', 'catv', 'choc', 'senc', 'obs', 'obsm', 'occutc', 'manv']
for col in veh_cols:
    df_vehicules[col] = df_vehicules[col].astype(str).apply(lambda x: x.split(',') if ',' in x else [x])

df_vehicules.head(20)

In [ ]:
# EXPLORATION

# 1️⃣ Compute length of each list cell for the vehicule columns
list_lengths_df = df_vehicules[veh_cols].map(
    lambda x: len(x) if isinstance(x, list) else 1
)

# 2️⃣ Compute how many distinct lengths there are per row
df_vehicules['nunique_lengths'] = list_lengths_df.nunique(axis=1)

# 3️⃣ (Optional) Store the max length — useful for padding later
df_vehicules['max_length'] = list_lengths_df.max(axis=1)

# 4️⃣ Identify misaligned rows (drifting)
misaligned_rows = df_vehicules[df_vehicules['nunique_lengths'] > 1]

print(f"⚠️ {len(misaligned_rows)} rows with misaligned list lengths detected.")

In [ ]:
# EXPLORATION

def pad_lists(row):
    max_len = row['max_length']
    for col in veh_cols:
        vals = row[col] if isinstance(row[col], list) else [row[col]]
        row[col] = (vals + [None] * (max_len - len(vals)))[:max_len]
    return row

df_vehicules = df_vehicules.apply(pad_lists, axis=1)

In [ ]:
# EXPLORATION

# Retirer les colonnes inutiles 'nunique_lengths', 'max_length' et 'senc'
df_vehicules = df_vehicules.drop(columns=['nunique_lengths', 'max_length', 'senc'])

df_vehicules.head(10)

In [ ]:
# EXPLORATION

# Make a copy to avoid modifying the original
df_veh_exploded = df_vehicules.copy()

# 🚀 Explode all list-type columns together (they all have equal-length lists now)
df_veh_exploded = df_veh_exploded.explode(veh_cols, ignore_index=True)

print(f"✓ Data exploded successfully: {len(df_veh_exploded):,} rows.")

# Ajout d'un identifiant unique pour chaque véhicule
df_veh_exploded = df_veh_exploded.reset_index(drop=True)
df_veh_exploded["vehicule_id"] = df_veh_exploded.index + 1

# Réorganisation des colonnes pour mettre l'id en premier
cols = ["vehicule_id"] + [c for c in df_veh_exploded.columns if c != "vehicule_id"]
df_veh_exploded = df_veh_exploded[cols]

df_veh_exploded.head(10)

In [ ]:
# Vérification des données après explosion
# df_veh_exploded.info()

# Each num_acc now has one row per vehicle
check = df_veh_exploded.groupby("num_acc")["num_veh"].nunique().describe()
print(check)

In [ ]:
# EXPLORATION

# Vérification des données après explosion

# Vérification des valeurs uniques sur les catégories
for col in ['num_veh', 'catv', 'choc', 'senc', 'obs', 'obsm', 'occutc', 'manv']:
    unique_values = df_veh_exploded[col].unique()
    print(f"\n Valeurs uniques dans '{col}': {unique_values}")

# Compter le nombre de 'nan' et '-1' dans chaque colonne catégorielle
for col in ['num_veh', 'catv', 'choc', 'senc', 'obs', 'obsm', 'occutc', 'manv']:
    nan_count = df_veh_exploded[col].isna().sum()
    neg_one_count = (df_veh_exploded[col] == '-1').sum()
    print(f"\n Colonne '{col}': NaN = {nan_count}, -1 = {neg_one_count}")

In [ ]:
# EXPLORATION

# Check des dtypes des colonnes
df_veh_exploded.dtypes

In [ ]:
# FONCTION 6 - TRANSFORMATION DF VEHICULES

# Check et nettoyage des colonnes catégorielles

# catv : retirer les parenthèses après "Voiturette"
# catv : 0 --> "Autre véhicule"
# catv : [50, 43, 42, 41, 60, 80] --> "Autre véhicule"
# catv : "Quad léger" & "Quad lourd" --> retirer les parenthèses
def clean_catv(value):
    if pd.isna(value):
        return value
    if value == '0':
        return 'Autre véhicule'
    if value in ['50', '43', '42', '41', '60', '80']:
        return 'Autre véhicule'
    if 'Voiturette (Quadricycle à moteur carrossé) (anciennement "voiturette ou tricycle à moteur")' in value:
        return 'Voiturette'
    if 'Quad léger <= 50 cm3 (Quadricycle à moteur non carrossé)' in value:
        return 'Quad léger <= 50 cm3'
    if 'Quad lourd > 50 cm3 (Quadricycle à moteur non carrossé)' in value:
        return 'Quad lourd > 50 cm3'
    return value

df_veh_exploded['catv'] = df_veh_exploded['catv'].apply(clean_catv)

# EXPLORATION
df_veh_exploded['catv'].value_counts(dropna=False)

In [ ]:
# FONCTION 6 - TRANSFORMATION DF VEHICULES

# clean : choc, obs, obsm, occutc, manv
# common rules to apply :
# '-1' --> NaN
# 'nan' or None --> np.nan
def clean_generic(value):
    if pd.isna(value) or value in ['nan', 'None']:
        return np.nan
    if value == '-1':
        return np.nan
    return value

for col in ['choc','obs', 'obsm', 'occutc', 'manv']:
    df_veh_exploded[col] = df_veh_exploded[col].apply(clean_generic)

# EXPLORATION
for col in ['choc', 'obs', 'obsm', 'occutc', 'manv']:
    print(f"\nColumn: {col}")
    print(df_veh_exploded[col].value_counts(dropna=False).head())

In [ ]:
# FONCTION 1 - Séparation en 5 dataframes

# 5. Usagers
df_usagers = df_source[[
    'sexe', 
    'grav', 
    'trajet', 
    'secu', 
    'secu_utl', 
    'catu', 
    'place', 
    'locp', 
    'actp', 
    'etatp', 
    'num_acc'
]].copy()

# EXPLORATION
df_usagers.head()
#df_usagers.info()

In [ ]:
# EXPLORATION

df_usagers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 11 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   sexe      475911 non-null  object
 1   grav      475911 non-null  object
 2   trajet    402358 non-null  object
 3   secu      413759 non-null  object
 4   secu_utl  413759 non-null  object
 5   catu      475911 non-null  object
 6   place     469630 non-null  object
 7   locp      101842 non-null  object
 8   actp      454084 non-null  object
 9   etatp     132740 non-null  object
 10  num_acc   475911 non-null  int64 
dtypes: int64(1), object(10)
memory usage: 39.9+ MB


In [ ]:
# EXPLORATION

# Test fonction explode_df pour df_usagers

# Définir les colonnes à exploser
cols_to_explode = ['sexe', 'grav', 'trajet', 'secu', 'secu_utl', 'catu', 'place', 'locp', 'actp', 'etatp']

df_usager_exploded = explode_df(df_usagers, cols_to_explode, 'usager_id')
df_usager_exploded.head()

⚠️ 385795 rows with misaligned list lengths detected — they will be padded.
✓ Data exploded successfully: 1,061,254 rows.


,usager_id,sexe,grav,trajet,secu,secu_utl,catu,place,locp,actp,etatp,num_acc,nunique_lengths,max_length
0,1,Masculin,Blessé,Promenade – loisirs,nan,nan,Conducteur,1,-1,-1,-1,201900020750,3,3
1,2,Masculin,Blessé,Promenade – loisirs,None,None,Passager,2,None,Se déplaçant,-1,201900020750,3,3
2,3,Masculin,Indemne,None,None,None,Conducteur,1,None,Se déplaçant,-1,201900020750,3,3
3,4,Masculin,Blessé,nan,nan,nan,Conducteur,1,-1,-1,-1,201900020796,1,1
4,5,Masculin,Indemne,Promenade – loisirs,nan,nan,Conducteur,1,nan,Se déplaçant,-1,201900020869,2,5


In [ ]:
# FONCTION 7 - TRANSFORMATION DF USAGERS

# Drop des colonnes inutiles 'nunique_lengths', 'max_length'
df_usager_exploded = df_usager_exploded.drop(columns=['nunique_lengths', 'max_length'])


# EXPLORATION

# Vérification des données après explosion
# Vérification des valeurs uniques sur les catégories
for col in ['sexe', 'grav', 'trajet', 'secu', 'secu_utl', 'catu', 'place', 'locp', 'actp', 'etatp']:
    unique_values = df_usager_exploded[col].unique()
    print(f"\n Valeurs uniques dans '{col}': {unique_values}")

# Compter le nombre de 'nan' et '-1' dans chaque colonne catégorielle
for col in ['sexe', 'grav', 'trajet', 'secu', 'secu_utl', 'catu', 'place', 'locp', 'actp', 'etatp']:
    nan_count = df_usager_exploded[col].isna().sum()
    neg_one_count = (df_usager_exploded[col] == '-1').sum()
    print(f"\n Colonne '{col}': NaN = {nan_count}, -1 = {neg_one_count}")


 Valeurs uniques dans 'sexe': ['Masculin' 'Féminin']

 Valeurs uniques dans 'grav': ['Blessé' 'Indemne' 'Tué']

 Valeurs uniques dans 'trajet': ['Promenade – loisirs' None 'nan' 'Utilisation professionnelle'
 'Domicile – travail' 'Courses – achats' 'Autre' 'Domicile – école' '-1']

 Valeurs uniques dans 'secu': ['nan' None 'Ceinture' 'Casque' 'Autre' 'Equipement réfléchissant'
 'Dispositif enfants']

 Valeurs uniques dans 'secu_utl': ['nan' None 'Oui' 'Non déterminable' 'Non']

 Valeurs uniques dans 'catu': ['Conducteur' 'Passager' 'Piéton' 'Piéton en roller ou en trottinette']

 Valeurs uniques dans 'place': ['1' '2' '9' '7' '3' '4' None 'nan' '5' '10' '8' '6']

 Valeurs uniques dans 'locp': ['-1' None 'nan' 'Sur contre allée'
 'Sur passage piéton - Avec signalisation lumineuse'
 'Sur passage piéton - Sans signalisation lumineuse'
 'Sur chaussée - A – 50 m du passage piéton'
 'Sur chaussée - A + 50 m du passage piéton' 'Sur trottoir'
 'Sur accotement' '9' 'Sur refuge ou BAU']

 Valeu

In [ ]:
# FONCTION 7 - TRANSFORMATION DF USAGERS

def clean_usager_df(df):
    df = df.copy()

    # 1️⃣ Normalize case and strip whitespace (avoids ' nan' or 'None ')
    df = df.apply(lambda col: col.astype(str).str.strip() if col.dtype == "object" else col)

    # 2️⃣ Replace 'nan', 'NaN', 'None', '-1' (string forms) with real np.nan
    df.replace(to_replace=['nan', 'NaN', 'None', '-1'], value=np.nan, inplace=True)

    # 3️⃣ Replace actual Python None or np.nan are already covered by the above
    # (no need to fillna, pandas already handles that)

    # 4️⃣ For 'locp' and 'actp' — digits, 'A', 'B' → np.nan
    df['locp'] = df['locp'].replace(r'^\d+$', np.nan, regex=True)
    df['actp'] = df['actp'].replace(r'^\d+$', np.nan, regex=True)
    df['actp'] = df['actp'].replace(['A', 'B'], np.nan)

    # 5️⃣ For accidents without pedestrians, nullify pedestrian-only columns
    mask_pieton_present = (
        df.groupby("num_acc")["catu"]
        .transform(lambda x: (x == "Piéton").any())
    )
    cols_to_null = ["locp", "actp", "etatp"]
    df.loc[~mask_pieton_present, cols_to_null] = np.nan

    return df


In [ ]:
# EXPLORATION

# Test clean_usager_df fonction
df_usager_clean = clean_usager_df(df_usager_exploded)
df_usager_clean.head()

,usager_id,sexe,grav,trajet,secu,secu_utl,catu,place,locp,actp,etatp,num_acc
0,1,Masculin,Blessé,Promenade – loisirs,NaN,NaN,Conducteur,1,NaN,NaN,NaN,201900020750
1,2,Masculin,Blessé,Promenade – loisirs,NaN,NaN,Passager,2,NaN,NaN,NaN,201900020750
2,3,Masculin,Indemne,NaN,NaN,NaN,Conducteur,1,NaN,NaN,NaN,201900020750
3,4,Masculin,Blessé,NaN,NaN,NaN,Conducteur,1,NaN,NaN,NaN,201900020796
4,5,Masculin,Indemne,Promenade – loisirs,NaN,NaN,Conducteur,1,NaN,NaN,NaN,201900020869


In [ ]:
# EXPLORATION

df_usager_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1061254 entries, 0 to 1061253
Data columns (total 12 columns):
 #   Column     Non-Null Count    Dtype 
---  ------     --------------    ----- 
 0   usager_id  1061254 non-null  int64 
 1   sexe       1061254 non-null  object
 2   grav       1061254 non-null  object
 3   trajet     778342 non-null   object
 4   secu       885685 non-null   object
 5   secu_utl   885685 non-null   object
 6   catu       1061254 non-null  object
 7   place      978364 non-null   object
 8   locp       84065 non-null    object
 9   actp       172739 non-null   object
 10  etatp      87841 non-null    object
 11  num_acc    1061254 non-null  int64 
dtypes: int64(2), object(10)
memory usage: 97.2+ MB


In [ ]:
# EXPLORATION

for col in ['sexe', 'grav', 'trajet', 'secu', 'secu_utl', 'catu', 'place', 'locp', 'actp', 'etatp']:
    print(f"\nColumn: {col}")
    print(df_usager_clean[col].value_counts(dropna=False).head())


Column: sexe
sexe
Masculin    719688
Féminin     341566
Name: count, dtype: int64

Column: grav
grav
Blessé     596739
Indemne    435945
Tué         28570
Name: count, dtype: int64

Column: trajet
trajet
Promenade – loisirs            406595
NaN                            282912
Domicile – travail             144371
Utilisation professionnelle    102249
Autre                           72909
Name: count, dtype: int64

Column: secu
secu
Ceinture              614903
Casque                194517
NaN                   175569
Autre                  58148
Dispositif enfants     15014
Name: count, dtype: int64

Column: secu_utl
secu_utl
Oui                 711856
NaN                 175569
Non déterminable    142429
Non                  31400
Name: count, dtype: int64

Column: catu
catu
Conducteur                            788583
Passager                              178561
Piéton                                 92477
Piéton en roller ou en trottinette      1633
Name: count, dtype: int64

Co

In [ ]:
# EXPLORATION

# Controller les valeurs uniques des catégories après nettoyage
for col in ['sexe', 'grav', 'trajet', 'secu', 'secu_utl', 'catu', 'place', 'locp', 'actp', 'etatp']:
    unique_values = df_usager_clean[col].unique()
    print(f"\n Valeurs uniques dans '{col}': {unique_values}")  


 Valeurs uniques dans 'sexe': ['Masculin' 'Féminin']

 Valeurs uniques dans 'grav': ['Blessé' 'Indemne' 'Tué']

 Valeurs uniques dans 'trajet': ['Promenade – loisirs' nan 'Utilisation professionnelle'
 'Domicile – travail' 'Courses – achats' 'Autre' 'Domicile – école']

 Valeurs uniques dans 'secu': [nan 'Ceinture' 'Casque' 'Autre' 'Equipement réfléchissant'
 'Dispositif enfants']

 Valeurs uniques dans 'secu_utl': [nan 'Oui' 'Non déterminable' 'Non']

 Valeurs uniques dans 'catu': ['Conducteur' 'Passager' 'Piéton' 'Piéton en roller ou en trottinette']

 Valeurs uniques dans 'place': ['1' '2' '9' '7' '3' '4' nan '5' '10' '8' '6']

 Valeurs uniques dans 'locp': [nan 'Sur contre allée'
 'Sur passage piéton - Avec signalisation lumineuse'
 'Sur passage piéton - Sans signalisation lumineuse'
 'Sur chaussée - A – 50 m du passage piéton'
 'Sur chaussée - A + 50 m du passage piéton' 'Sur trottoir'
 'Sur accotement' 'Sur refuge ou BAU']

 Valeurs uniques dans 'actp': [nan 'Se déplaçant' 'Trav

In [71]:
# Dataframe usagers prêt à l'export

df_usager_clean.head(10)

,usager_id,sexe,grav,trajet,secu,secu_utl,catu,place,locp,actp,etatp,num_acc
0,1,Masculin,Blessé,Promenade – loisirs,NaN,NaN,Conducteur,1,NaN,NaN,NaN,201900020750
1,2,Masculin,Blessé,Promenade – loisirs,NaN,NaN,Passager,2,NaN,NaN,NaN,201900020750
2,3,Masculin,Indemne,NaN,NaN,NaN,Conducteur,1,NaN,NaN,NaN,201900020750
3,4,Masculin,Blessé,NaN,NaN,NaN,Conducteur,1,NaN,NaN,NaN,201900020796
4,5,Masculin,Indemne,Promenade – loisirs,NaN,NaN,Conducteur,1,NaN,NaN,NaN,201900020869
5,6,Masculin,Indemne,Promenade – loisirs,NaN,NaN,Passager,9,NaN,NaN,NaN,201900020869
6,7,Masculin,Blessé,Promenade – loisirs,NaN,NaN,Conducteur,1,NaN,NaN,NaN,201900020869
7,8,Masculin,Indemne,Promenade – loisirs,NaN,NaN,Passager,7,NaN,NaN,NaN,201900020869
8,9,Masculin,Indemne,Promenade – loisirs,NaN,NaN,Passager,2,NaN,NaN,NaN,201900020869
9,10,Masculin,Blessé,NaN,NaN,NaN,Conducteur,1,NaN,NaN,NaN,201900021309


# ZOUBIR